# 📝 [제 7편] 선형 회귀 (3) - 파이썬으로 직접 학습시키기

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「Linear Regression (III) - simple • multi-variable example」 (ML_LEC_15)

---

## 🎯 학습 목표
1. 선형 회귀를 행렬 식 Y = X · W + b 로 나타내고 각 행렬의 형상을 설명할 수 있다.
2. 수치미분과 경사하강법으로 W, b 를 학습시키는 코드를 단계별로 완성할 수 있다.
3. 입력 변수가 여러 개인 다변수 선형 회귀로 미래 값을 예측할 수 있다.

---

## 🧭 지난 시간 복습
* 제 2편 수치미분 + 제 3편 행렬 + 제 5편 손실함수 + 제 6편 경사하강법 → **오늘 전부 합쳐서 진짜 인공지능을 학습시킵니다!**

[⬅ 이전 : 제 6편 선형 회귀 (2) - 경사하강법](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/06_gradient_descent.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 8편 로지스틱 회귀 - 합격일까 불합격일까? (분류) ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/08_logistic_regression.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

## 📖 개념 다지기 1. 구현 설계도

| 순서 | 수학 | 파이썬 |
|:-:|:---|:---|
| 1 | 학습 데이터 (x, t) | `x_data`, `t_data` (numpy 행렬) |
| 2 | y = Wx + b | `W = np.random.rand(...)`, `b = np.random.rand(...)` 임의의 값으로 시작 |
| 3 | E(W, b) = (1/n) × Σ {t<sub>i</sub> − (W x<sub>i</sub> + b)}² | `loss_func(x, t)` |
| 4 | 학습률 α | `learning_rate = 1e-2` 등 |
| 5 | W ← W − α × ∂E/∂W,  b ← b − α × ∂E/∂b | `W -= learning_rate * numerical_derivative(f, W)` 를 반복 |

### 행렬로 한꺼번에 계산하기
데이터가 5개이면 y = Wx + b 를 5번 계산해야 합니다. 행렬 곱을 쓰면 **한 번에** 계산됩니다.

　X · W + b = Y 　→ 　(5 × 1) · (1 × 1) = (5 × 1)

　( 1 )　　　　　　 ( 1W + b )
　( 2 )　　　　　　 ( 2W + b )
　( 3 ) · ( W ) + b = ( 3W + b )
　( 4 )　　　　　　 ( 4W + b )
　( 5 )　　　　　　 ( 5W + b )

> **[연습 1]** 입력이 25개 데이터 × 3개 변수인 X (25 × 3) 에 대해 Y 가 (25 × 1) 이 되려면 W 의 형상은 ( 　 × 　 ) 이다.

In [ ]:
#@title ▶ 수치미분 함수 (제 2편에서 만든 것, 실행만 하세요)
def numerical_derivative(f, x):
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x)
        x[idx] = float(tmp_val) - delta_x
        fx2 = f(x)
        grad[idx] = (fx1 - fx2) / (2 * delta_x)
        x[idx] = tmp_val
        it.iternext()
    return grad
print('numerical_derivative 준비 완료')

## 💻 실습 1. 단순 선형 회귀 (입력 변수 1개)

| 입력 x | 1 | 2 | 3 | 4 | 5 |
|:-:|:-:|:-:|:-:|:-:|:-:|
| 정답 t | 2 | 3 | 4 | 5 | 6 |

표를 보면 t = x + 1 입니다. 인공지능이 **스스로 W = 1, b = 1 을 찾아내는지** 확인해 봅시다!

### [1] 학습 데이터 준비

In [ ]:
x_data = np.array([1, 2, 3, 4, 5]).reshape(5, 1)
t_data = np.array([2, 3, 4, 5, 6]).reshape(5, 1)
print('x_data.shape =', x_data.shape, ', t_data.shape =', t_data.shape)

### [2] 임의의 직선 y = Wx + b (W, b 를 아무 값으로 시작)
`np.random.rand` 는 0 ~ 1 사이의 수를 무작위로 만듭니다. (모두 같은 결과가 나오도록 `seed` 를 고정)

In [ ]:
np.random.seed(42)
W = np.random.rand(1, 1)
b = np.random.rand(1)
print('처음 W =', W, ', W.shape =', W.shape, ' / 처음 b =', b, ', b.shape =', b.shape)

### [3] 손실함수, 오차 확인 함수, 예측 함수

In [ ]:
def loss_func(x, t):
    y = np.dot(x, W) + b                       # 행렬 곱으로 모든 데이터의 y 를 한 번에
    return np.sum((t - y)**2) / len(x)         # 오차² 의 평균

def error_val(x, t):                           # 학습 중 손실 값을 확인하는 용도
    y = np.dot(x, W) + b
    return np.sum((t - y)**2) / len(x)

def predict(x):                                # 학습이 끝난 뒤 미래 값 예측
    return np.dot(x, W) + b

print('처음 손실 값 =', error_val(x_data, t_data))

### [4] 학습 - 경사하강법 반복

```python
f = lambda x : loss_func(x_data, t_data)
```
❓ 입력 x 를 쓰지도 않는데 왜 이렇게 만들까?
→ `numerical_derivative(f, W)` 는 **W 안의 값을 직접 ±Δx 만큼 바꾼 뒤** f 를 부릅니다. loss_func 는 바깥의 W 를 그대로 쓰므로, 바뀐 W 로 손실이 계산됩니다.
　즉 f 는 "**현재 W, b 로 계산한 손실 값**" 을 돌려주는 함수입니다.

In [ ]:
learning_rate = 1e-2
f = lambda x : loss_func(x_data, t_data)

history = [(0, W[0, 0], b[0], error_val(x_data, t_data))]   # 애니메이션용 기록 (처음 상태 포함)
print('처음 손실 =', error_val(x_data, t_data), ' W =', W.ravel(), ' b =', b)
for step in range(8001):
    W -= learning_rate * numerical_derivative(f, W)
    b -= learning_rate * numerical_derivative(f, b)
    if step < 60 or step % 40 == 0:
        history.append((step + 1, W[0, 0], b[0], error_val(x_data, t_data)))
    if step % 1000 == 0:
        print(f'step {step:5d} :  손실 = {error_val(x_data, t_data):.10f},  W = {W[0,0]:.6f},  b = {b[0]:.6f}')

### [5] 결과 확인과 예측

In [ ]:
print('학습 결과 :  W =', W.ravel(), ', b =', b)
print('공부 시간 43 의 예측값 =', predict(np.array([[43]])))

> **[확인 1]** 학습 후 W ≒ ( 　 ), b ≒ ( 　 ) 이고, x = 43 의 예측값은 ( 　 ) 이다. t = x + 1 과 일치하나요?

## 🎬 애니메이션 1. 직선이 데이터에 맞춰지는 과정

위의 학습 기록(`history`)을 그대로 재생합니다. 왼쪽은 직선, 오른쪽은 손실 값의 변화(로그 눈금)입니다.

In [ ]:
#@title 🎬 애니메이션 1 실행 (반드시 실습 1 을 먼저 실행하세요)
hist = np.array(history)
sel = list(range(0, 60)) + list(range(60, len(hist), 6)) + [len(hist)-1]*6
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.5))
def draw(f):
    k = sel[f]; st, w, bb, e = hist[k]
    axL.clear(); axR.clear()
    axL.scatter(x_data, t_data, s=80, color='royalblue', zorder=5, label='학습 데이터')
    xs = np.linspace(0, 6, 10)
    axL.plot(xs, w*xs + bb, color='crimson', lw=2.5, label=f'y = {w:.3f}x + {bb:.3f}')
    axL.set_xlim(0, 6); axL.set_ylim(0, 8); axL.grid(alpha=0.3); axL.legend(loc='upper left')
    axL.set_title(f'step {int(st)}', fontsize=14)
    axR.semilogy(hist[:k+1, 0], hist[:k+1, 3], color='purple', lw=2)
    axR.set_xlim(0, hist[-1, 0]); axR.set_ylim(hist[:, 3].min()*0.5, hist[:, 3].max()*2)
    axR.set_xlabel('step'); axR.set_ylabel('손실 값 (로그 눈금)'); axR.grid(alpha=0.3, which='both')
    axR.set_title(f'손실 = {e:.2e}', fontsize=14)
anim = animation.FuncAnimation(fig, draw, frames=len(sel), interval=120)
plt.close(fig)
show_anim(anim)

> **[관찰 1]** 손실 값은 처음에 ( 빠르게 / 느리게 ) 줄어들다가 나중에는 ( 빠르게 / 느리게 ) 줄어든다.
> 직선은 대략 step ( 　　 ) 쯤부터 데이터와 거의 겹친다.

## 💻 실습 2. 다변수 선형 회귀 (입력 변수 3개)

1차, 2차, 3차 시험 점수(x₁, x₂, x₃)로 최종 점수(t)를 예측합니다. (학생 25명 데이터, 원본 강의의 data-01-test-score.csv)

　y = W₁x₁ + W₂x₂ + W₃x₃ + b 　→ 　X · W + b = Y,　(25 × 3) · (3 × 1) = (25 × 1)

### [1] 데이터 준비

In [ ]:
csv_text = """73,80,75,152
93,88,93,185
89,91,90,180
96,98,100,196
73,66,70,142
53,46,55,101
69,74,77,149
47,56,60,115
87,79,90,175
79,70,88,164
69,70,73,141
70,65,74,141
93,95,91,184
79,80,73,152
70,73,78,148
93,89,96,192
78,75,68,147
81,90,93,183
88,92,86,177
78,83,77,159
82,86,90,177
86,82,89,175
78,83,85,175
76,83,71,149
96,93,95,192"""
with open('data-01-test-score.csv', 'w') as fp:
    fp.write(csv_text)

loaded_data = np.loadtxt('./data-01-test-score.csv', delimiter=',', dtype=np.float32)
x_data = loaded_data[:, 0:-1]
t_data = loaded_data[:, [-1]]
print('x_data.shape =', x_data.shape, ', t_data.shape =', t_data.shape)
print(loaded_data[:5], '...')

### [2] ~ [4] 초기화와 학습

⚠️ 점수(70 ~ 100)처럼 입력 값이 크면 기울기도 커지므로 학습률을 **아주 작게 (1e-5)** 해야 발산하지 않습니다.
(직접 `1e-3` 으로 바꿔서 실행해 보면 손실 값이 어떻게 되는지 확인해 보세요!)

In [ ]:
np.random.seed(0)
W = np.random.rand(3, 1)          # 변수 3개 → W 는 (3 × 1)
b = np.random.rand(1)

learning_rate = 1e-5
f = lambda x : loss_func(x_data, t_data)

loss_hist, W_hist = [], []
print('처음 손실 =', error_val(x_data, t_data))
for step in range(10001):
    W -= learning_rate * numerical_derivative(f, W)
    b -= learning_rate * numerical_derivative(f, b)
    if step % 100 == 0:
        loss_hist.append(error_val(x_data, t_data)); W_hist.append((W.copy(), b.copy()))
    if step % 2000 == 0:
        print(f'step {step:5d} : 손실 = {error_val(x_data, t_data):.4f},  W = {W.ravel()},  b = {b}')

### [5] 결과와 예측

In [ ]:
print('W =', W.ravel(), ', b =', b)
test = np.array([100, 98, 81])
print('1차 100, 2차 98, 3차 81 점인 학생의 최종 점수 예측 =', predict(test))

> **[확인 2]** 학습된 W₁, W₂, W₃ 중 가장 큰 값은 ( 　 ) 이다. 이것은 ( 1차 / 2차 / 3차 ) 시험이 최종 점수에 가장 큰 영향을 준다는 뜻이다.
> 1차 100, 2차 98, 3차 81 점 학생의 최종 점수 예측값은 약 ( 　　 ) 점이다.

## 🎬 애니메이션 2. 예측이 정답에 가까워지는 과정

가로축은 **정답 t**, 세로축은 **예측 y** 입니다. 예측이 완벽하면 모든 점이 대각선 y = t 위에 놓입니다.

In [ ]:
#@title 🎬 애니메이션 2 실행 (반드시 실습 2 를 먼저 실행하세요)
sel = list(range(0, 20)) + list(range(20, len(W_hist), 3)) + [len(W_hist)-1]*6
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.8))
def draw(f):
    k = sel[f]; Wk, bk = W_hist[k]
    y = x_data @ Wk + bk
    axL.clear(); axR.clear()
    axL.plot([90, 210], [90, 210], 'k--', lw=1, label='y = t (완벽한 예측)')
    axL.scatter(t_data, y, s=50, color='crimson', alpha=0.8, label='학생 25명')
    axL.set_xlim(90, 210); axL.set_ylim(90, 210); axL.set_aspect('equal'); axL.grid(alpha=0.3)
    axL.set_xlabel('정답 t'); axL.set_ylabel('예측 y'); axL.legend(loc='upper left')
    axL.set_title(f'step {k*100}   W = {np.round(Wk.ravel(), 2)}', fontsize=12)
    axR.plot(np.arange(k+1)*100, loss_hist[:k+1], color='purple', lw=2)
    axR.set_xlim(0, 10000); axR.set_ylim(0, max(loss_hist)*1.05); axR.grid(alpha=0.3)
    axR.set_xlabel('step'); axR.set_ylabel('손실 값'); axR.set_title(f'손실 = {loss_hist[k]:.2f}', fontsize=13)
anim = animation.FuncAnimation(fig, draw, frames=len(sel), interval=120)
plt.close(fig)
show_anim(anim)

## ✏️ 연습 문제

> **[문제 1]** 아래 데이터는 t = 2x − 1 을 따릅니다. 실습 1 의 코드를 그대로 이용해 학습시키고, W 와 b 가 각각 2, −1 에 가까워지는지 확인하세요. 그리고 x = 10 일 때의 예측값을 구하세요.

| x | 1 | 2 | 3 | 4 | 5 |
|:-:|:-:|:-:|:-:|:-:|:-:|
| t | 1 | 3 | 5 | 7 | 9 |

> **[문제 2]** 학습률을 1e-1, 1e-2, 1e-3 으로 바꿔 각각 2000번만 학습시켰을 때 손실 값을 비교하세요. 어떤 학습률이 가장 좋았나요? ( 　　 )

In [ ]:
x_data = np.array([1, 2, 3, 4, 5]).reshape(5, 1)
t_data = np.array(______).reshape(5, 1)

np.random.seed(1)
W = np.random.rand(1, 1); b = np.random.rand(1)
learning_rate = ______
f = lambda x : loss_func(x_data, t_data)
for step in range(______):
    W -= learning_rate * numerical_derivative(f, W)
    b -= learning_rate * numerical_derivative(f, b)

print('W =', W.ravel(), ', b =', b, ', 손실 =', error_val(x_data, t_data))
print('x = 10 의 예측값 =', predict(np.array([[10]])))

In [ ]:
#@title 🔑 정답 확인 (먼저 풀어 본 뒤에 실행하세요) { display-mode: "form" }
x_data = np.array([1, 2, 3, 4, 5]).reshape(5, 1); t_data = np.array([1, 3, 5, 7, 9]).reshape(5, 1)
f = lambda x : loss_func(x_data, t_data)
for lr in [1e-1, 1e-2, 1e-3]:
    np.random.seed(1); W = np.random.rand(1, 1); b = np.random.rand(1)
    for step in range(2000):
        W -= lr * numerical_derivative(f, W); b -= lr * numerical_derivative(f, b)
    print(f'학습률 {lr:g} : W = {W[0,0]:.4f}, b = {b[0]:.4f}, 손실 = {error_val(x_data, t_data):.2e}, x=10 예측 = {predict(np.array([[10]]))[0,0]:.3f}')
print('→ 1e-1 은 너무 커서 출렁이며 학습이 안 되고(손실이 큼), 1e-2 가 가장 좋고, 1e-3 은 아직 덜 학습됨. 정답: W = 2, b = −1, 예측 19')

## 📌 정리

| 단계 | 코드 |
|:---|:---|
| 데이터 | `x_data = loaded_data[:, 0:-1]`, `t_data = loaded_data[:, [-1]]` |
| 초기화 | `W = np.random.rand(변수 개수, 1)`, `b = np.random.rand(1)` |
| 손실함수 | `np.sum((t - (np.dot(x, W) + b))**2) / len(x)` |
| 학습 | `W -= learning_rate * numerical_derivative(f, W)` 반복 |
| 예측 | `np.dot(새 입력, W) + b` |

### 🏆 도전 과제
선형 회귀는 **숫자**(점수)를 예측합니다. 그런데 "합격 / 불합격" 처럼 **두 가지 중 하나**를 맞히려면 y = Wx + b 를 어떻게 바꿔야 할까요?
힌트: 출력이 항상 0 과 1 사이가 되는 함수가 있다면…?  → 다음 시간: **로지스틱 회귀**
(                                                                                                                            )

---

[⬅ 이전 : 제 6편 선형 회귀 (2) - 경사하강법](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/06_gradient_descent.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 8편 로지스틱 회귀 - 합격일까 불합격일까? (분류) ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/08_logistic_regression.ipynb)